# Misclassification Gallery

A model's mistakes are often the best teaching examples. This notebook finds MNIST images the CNN gets wrong and overlays simple Grad-CAM heatmaps.

## Initialize PyTorch

- Import PyTorch, torchvision, functional helpers, and plotting tools
- Seed PyTorch for reproducibility
- Detect device (GPU, Apple Silicon, CPU)

In [ ]:
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

torch.manual_seed(7)

def get_device():
    if torch.cuda.is_available():
        return "cuda"
    if torch.backends.mps.is_available():
        return "mps"
    return "cpu"

device = get_device()

## Define CNN Model

- Build a CNN with named convolution layers
- Keep the final convolution layer available for Grad-CAM
- Output scores for the ten digit classes

In [ ]:
class MNISTCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 8, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(8, 16, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2)
        self.relu = nn.ReLU()
        self.classifier = nn.Sequential(nn.Flatten(), nn.Linear(16 * 7 * 7, 64), nn.ReLU(), nn.Linear(64, 10))

    def forward(self, x):
        x = self.pool(self.relu(self.conv1(x)))
        x = self.pool(self.relu(self.conv2(x)))
        return self.classifier(x)

model = MNISTCNN().to(device)

## Load Data and Train

- Load MNIST train and test data
- Train briefly on a subset so some mistakes remain
- Use a larger test subset to search for incorrect predictions

In [ ]:
transform = transforms.ToTensor()
train_data = datasets.MNIST(root="../../data", train=True, download=True, transform=transform)
test_data = datasets.MNIST(root="../../data", train=False, download=True, transform=transform)

train_loader = DataLoader(Subset(train_data, range(2500)), batch_size=64, shuffle=True)
test_loader = DataLoader(Subset(test_data, range(2000)), batch_size=1)

loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

model.train()
for images, labels in train_loader:
    images, labels = images.to(device), labels.to(device)
    loss = loss_fn(model(images), labels)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

## Define Grad-CAM

- Capture target-layer activations and gradients
- Generate a class-specific heatmap
- Return a normalized heatmap for overlaying on the digit image

In [ ]:
def grad_cam(model, image, target_class):
    activations = None
    gradients = None

    def forward_hook(module, inputs, output):
        nonlocal activations
        activations = output

    def backward_hook(module, grad_input, grad_output):
        nonlocal gradients
        gradients = grad_output[0]

    forward_handle = model.conv2.register_forward_hook(forward_hook)
    backward_handle = model.conv2.register_full_backward_hook(backward_hook)

    model.zero_grad()
    logits = model(image)
    logits[:, target_class].backward()

    weights = gradients.mean(dim=(2, 3), keepdim=True)
    cam = (weights * activations).sum(dim=1, keepdim=True)
    cam = F.relu(cam)
    cam = F.interpolate(cam, size=(28, 28), mode="bilinear", align_corners=False)
    cam = cam.squeeze().detach().cpu()
    cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)

    forward_handle.remove()
    backward_handle.remove()
    return cam

## Collect Mistakes

- Run the model over test images
- Find examples where prediction differs from the true label
- Store true label, predicted label, confidence, image, and heatmap
- Stop after collecting a small gallery

In [ ]:
model.eval()
mistakes = []

for image, label in test_loader:
    image = image.to(device)
    with torch.no_grad():
        probabilities = F.softmax(model(image), dim=1).squeeze().cpu()
        prediction = probabilities.argmax().item()
        confidence = probabilities[prediction].item()

    if prediction != label.item():
        heatmap = grad_cam(model, image, prediction)
        mistakes.append((image.cpu().squeeze(), heatmap, label.item(), prediction, confidence))

    if len(mistakes) == 8:
        break

len(mistakes)

## Plot Mistake Gallery

- Display misclassified digits in a grid
- Overlay Grad-CAM heatmaps on each mistake
- Show true label, predicted label, and confidence

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(10, 5))
for ax, (image, heatmap, label, prediction, confidence) in zip(axes.ravel(), mistakes):
    ax.imshow(image, cmap="gray")
    ax.imshow(heatmap, cmap="magma", alpha=0.45)
    ax.set_title(f"true={label} pred={prediction}\nconf={confidence:.2f}")
    ax.axis("off")

plt.tight_layout()

## Exercise

Train for more epochs and rerun the gallery. Which mistakes disappear, and which mistakes remain?